# 🖼️ Excel Image Auto-Filler
### How to use:
1. Click **Runtime → Run All**
2. Scroll to **Step 3**, upload your `.xlsx` file
3. Wait, then click **Download**

✅ That's it!

In [ ]:
#@title ⚙️ Step 1 — Install everything (runs automatically, ~1 min)
import subprocess, sys
print('📦 Installing libraries...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'playwright', 'openpyxl', 'Pillow'], check=True)
print('🌐 Installing Playwright browser (Chromium)...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'playwright', 'install-deps', 'chromium'],
               capture_output=True)
print('✅ All ready!')

In [ ]:
#@title ⚙️ Step 2 — Load the automation code (runs automatically)
import os, re, time, shutil, tempfile, urllib.parse, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from playwright.sync_api import sync_playwright
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import column_index_from_string
from PIL import Image as PILImage

MAX_WORKERS = 3
print_lock = threading.Lock()

def log(msg):
    with print_lock:
        print(msg)

def extract_query(cell_text):
    """Clean up cell text into a good search query."""
    text = str(cell_text)
    # Handle BOTH actual newlines AND literal \n text in cells
    text = text.replace('\\n', '\n')
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    # Join all lines with pipe separator
    q = ' | '.join(lines) if lines else text.strip()
    # Remove curly quotes
    q = re.sub(r"['\u2018\u2019\u201c\u201d]", '', q)
    return q[:200]

def screenshot_one(query, dest, wid):
    """Open Bing Images, wait for the grid to fully load, screenshot it."""
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=True,
                args=['--no-sandbox', '--disable-dev-shm-usage',
                      '--disable-gpu', '--disable-setuid-sandbox',
                      '--window-size=1300,950']
            )
            page = browser.new_page(
                viewport={'width': 1300, 'height': 950},
                user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
            )

            encoded = urllib.parse.quote(query)
            url = f'https://www.bing.com/images/search?q={encoded}&form=HDRSC2&first=1'
            page.goto(url, wait_until='networkidle', timeout=30000)
            page.wait_for_timeout(3000)

            # Scroll down slightly so thumbnails render
            page.evaluate('window.scrollTo(0, 150)')
            page.wait_for_timeout(2000)

            # Wait for the image grid to appear
            try:
                page.wait_for_selector('img.mimg', timeout=8000)
            except:
                pass

            # Screenshot the image results grid specifically
            captured = False
            for sel in ['ul.dgControl_list', 'div.dgControl', '#mmComponent_images_1', 'div#b_results']:
                try:
                    el = page.query_selector(sel)
                    if el:
                        el.screenshot(path=dest)
                        captured = True
                        break
                except:
                    continue

            if not captured:
                # Full page screenshot as last resort
                page.screenshot(path=dest, full_page=False)

            browser.close()

        ok = os.path.exists(dest) and os.path.getsize(dest) > 5000
        if ok:
            with PILImage.open(dest) as im:
                im.load()
            log(f'  [w{wid}] 📸 {query[:50]}')
        else:
            log(f'  [w{wid}] ✗ empty screenshot: {query[:40]}')
        return ok

    except Exception as e:
        log(f'  [w{wid}] ✗ {str(e).splitlines()[0][:80]}')
        return False

def worker(task):
    task['success'] = screenshot_one(task['query'], task['img_path'], task['wid'])
    return task

def embed_image(ws, img_path, cell_addr):
    try:
        compressed = str(Path(img_path).with_suffix('.jpg'))
        with PILImage.open(img_path) as im:
            im = im.convert('RGB')
            im.save(compressed, 'JPEG', quality=85, optimize=True)
            w, h = im.size
        xl = XLImage(compressed)
        xl.anchor = cell_addr
        xl.width = w
        xl.height = h
        ws.add_image(xl)
        row_num = int(re.sub(r'[A-Za-z]', '', cell_addr))
        ws.row_dimensions[row_num].height = max(80, h * 0.75)
        return True
    except Exception as e:
        log(f'  ⚠ Embed error {cell_addr}: {e}')
        return False

def run_process(filepath, col_read='A', col_write='B', start_row=1):
    print(f'\n📂 Opening: {filepath}')
    wb = load_workbook(filepath)
    ws = wb.active
    if ws is None:
        ws = wb[wb.sheetnames[0]]
    print(f'📋 Sheet: {ws.title}')

    read_idx = column_index_from_string(col_read.upper())
    write_col = col_write.upper()
    rows = []
    for row in ws.iter_rows(min_row=start_row, min_col=read_idx, max_col=read_idx):
        cell = row[0]
        if cell.value and str(cell.value).strip():
            rows.append((cell.row, str(cell.value).strip()))

    if not rows:
        print('⚠  No data found in column', col_read)
        return None

    print(f'🔍 Found {len(rows)} rows.')

    # Show a preview of how queries will look
    print('\n🔎 Search queries preview:')
    for row_num, raw in rows[:3]:
        print(f'  Row {row_num}: "{extract_query(raw)}"')
    if len(rows) > 3:
        print(f'  ... and {len(rows)-3} more')

    print(f'\n⚡ Running {min(MAX_WORKERS, len(rows))} at a time in parallel...\n')

    ws.column_dimensions[write_col].width = 60
    tmp_dir = tempfile.mkdtemp(prefix='xl_shots_')

    tasks = []
    for idx, (row_num, raw_text) in enumerate(rows):
        uid = f'{row_num}_{int(time.time()*1000)%100000}_{idx}'
        tasks.append({
            'row_num':   row_num,
            'query':     extract_query(raw_text),
            'img_path':  os.path.join(tmp_dir, f'shot_{uid}.png'),
            'cell_addr': f'{write_col}{row_num}',
            'wid':       idx,
        })

    t0 = time.time()
    completed = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(worker, t): t for t in tasks}
        for fut in as_completed(futures):
            try:
                completed.append(fut.result())
            except Exception as e:
                t = futures[fut]
                t['success'] = False
                completed.append(t)

    print(f'\n⏱  Screenshots done in {time.time()-t0:.1f}s')
    print('🖼  Embedding into Excel...\n')

    ok_count = 0
    for task in sorted(completed, key=lambda t: t['row_num']):
        cell_addr = task['cell_addr']
        if task['success'] and os.path.exists(task['img_path']) and \
           embed_image(ws, task['img_path'], cell_addr):
            print(f'  ✅ {cell_addr}  ←  {task["query"][:60]}')
            ok_count += 1
        else:
            ws[cell_addr] = '(screenshot failed)'
            print(f'  ✗  {cell_addr}  failed')

    out = filepath.replace('.xlsx', '_with_images.xlsx')
    if out == filepath:
        out = filepath + '_with_images.xlsx'
    wb.save(out)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f'\n✅ Done! {ok_count}/{len(rows)} images embedded.')
    return out

print('✅ Code loaded!')

In [ ]:
#@title 📁 Step 3 — Upload your Excel file and Run!
from google.colab import files
from IPython.display import display, HTML

print('👇 Click "Choose Files" below to upload your .xlsx file')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f'\n✅ Uploaded: {filename}')

    #@markdown ### ⚙️ Settings (only change if needed)
    col_read  = 'A'  #@param {type:"string"}
    col_write = 'B'  #@param {type:"string"}
    start_row =  1   #@param {type:"integer"}

    print(f'\n🚀 Starting...\n')
    output_file = run_process(filename,
                              col_read=col_read,
                              col_write=col_write,
                              start_row=start_row)

    if output_file and os.path.exists(output_file):
        display(HTML('<br><h3 style="color:green">✅ Done! Click below to download:</h3>'))
        files.download(output_file)
    else:
        print('⚠️ Something went wrong.')
else:
    print('⚠️ No file uploaded.')